In [1]:
!pip install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()
# Browse and select multiple CSV files
uploaded = files.upload()
# Load selected CSV files into PySpark DataFrames
dataframes = {}
for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df
    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv

File: lazy_execution_stores.csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000.csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+-----------

In [3]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000.csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|

In [4]:
#transformation
completed_df=sales_df.filter(F.col("order_status")=="Completed")

print("transforation created")

#action 1
print("completed orders:",completed_df.count())

#action2
completed_df.show(5)

transforation created
completed orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
+--------+--------

In [8]:
#cache
completed_df.cache()

print("Is Cached",completed_df.is_cached)

completed_df.count()

completed_df.show()

completed_df.groupBy("product_id").count().show()

Is Cached True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8

In [13]:
high_quantity_df=sales_df.filter(F.col("quantity")>5)

high_quantity_df.persist(StorageLevel.MEMORY_AND_DISK)

print(high_quantity_df.count())

high_quantity_df.show(10)

print(high_quantity_df.storageLevel)



11087
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      29

In [12]:
# Check cache status
print("Before:", completed_df.is_cached)

# Remove cache
completed_df.unpersist(blocking=True)

# Check again
print("After:", completed_df.is_cached)

# Remove the other persisted DataFrame
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [14]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)
















+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [15]:

# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 1.038 seconds
Second Execution: 0.157 seconds


In [16]:
print("Total_records:",sales_df.count())

print("Number of Partitions",sales_df.rdd.getNumPartitions())



Total_records: 25000
Number of Partitions 1


In [18]:
#create 8 partitions
partitioned_df=sales_df.repartition(8)

print("Original",sales_df.rdd.getNumPartitions())

print("Number of Partitions",partitioned_df.rdd.getNumPartitions())

partitioned_df.count()

#

Original 1
Number of Partitions 8


25000

In [19]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [20]:
# Partition data by store_id
store_partitioned_df = sales_df.repartition(
    8,
    "store_id"
)

print("Partitions:",
      store_partitioned_df.rdd.getNumPartitions())

# Inspect partition distribution
store_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id", "store_id"
).count().orderBy(
    "partition_id", "store_id"
).show(30)

Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [21]:
# Create 8 partitions
df = sales_df.repartition(8)

# Reduce using repartition
df_repartition = df.repartition(3)

# Reduce using coalesce
df_coalesce = df.coalesce(3)

print("Repartition:",
      df_repartition.rdd.getNumPartitions())

print("Coalesce:",
      df_coalesce.rdd.getNumPartitions())

# Examine execution plans
print("REPARTITION PLAN")
df_repartition.explain()

print("COALESCE PLAN")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1379]
         +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1379]
   +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Fo

Exercise partition

In [24]:
print("Total Records in sales_df:", sales_df.count())
print("Number of Partitions in sales_df:", sales_df.rdd.getNumPartitions())

Total Records in sales_df: 25000
Number of Partitions in sales_df: 1


In [25]:
#create 8 partitions
partitioned_df=sales_df.repartition(12)

print("Original",sales_df.rdd.getNumPartitions())

print("Number of Partitions",partitioned_df.rdd.getNumPartitions())

partitioned_df.count()


Original 1
Number of Partitions 12


25000

In [26]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [28]:
# Partition data by store_id
store_partitioned_df = sales_df.repartition(
    8,
    "product_id"
)

print("Partitions:",
      store_partitioned_df.rdd.getNumPartitions())

# Inspect partition distribution
store_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id", "product_id"
).count().orderBy(
    "partition_id", "product_id"
).show(30)

Partitions: 8
+------------+----------+-----+
|partition_id|product_id|count|
+------------+----------+-----+
|           0|      P004| 1574|
|           0|      P011| 1716|
|           0|      P014| 1684|
|           1|      P009| 1654|
|           3|      P001| 1743|
|           3|      P008| 1596|
|           4|      P002| 1613|
|           4|      P006| 1678|
|           4|      P010| 1695|
|           4|      P015| 1645|
|           5|      P003| 1676|
|           5|      P005| 1696|
|           5|      P007| 1602|
|           6|      P012| 1691|
|           6|      P013| 1737|
+------------+----------+-----+



In [29]:
# Create 8 partitions
df = sales_df.repartition(12)

# Reduce using repartition
df_repartition = df.repartition(4)

# Reduce using coalesce
df_coalesce = df.coalesce(4)

print("Repartition:",
      df_repartition.rdd.getNumPartitions())

print("Coalesce:",
      df_coalesce.rdd.getNumPartitions())


Repartition: 4
Coalesce: 4
